# nb9.4 — Poster Layout Engine: 48×36 PDF Posters from a Parameter Dict

*Week 9, Chapter 9.4. Companion notebook.*

The poster session is the part of the symposium where you cannot hide behind a slide deck. Your work
is pinned to a four-by-three-foot board, and forty people walk past it in two hours. Most of them stop
for twelve seconds; a few stop for two minutes. The poster has to do two jobs simultaneously: catch
the twelve-second skimmer with a single readable headline, and reward the two-minute reader with the
identification design, the table, and the figure they came for.

Devon learned the hard way that this is not a Photoshop job. He spent six hours in PowerPoint last
spring trying to align panels by eye, gave up at midnight, and printed a poster whose figure had the
wrong axis labels because he had cropped a screenshot instead of regenerating from the analysis. The
worst part is that the *design* of his poster — what panel goes where — was fine; what failed was the
*production*: a hand-laid-out poster is a graveyard of misaligned panels and out-of-date figures.

The fix is to treat the poster as a *program*. You write a parameter dict — what goes in each panel,
how big it is, what font — and a layout engine in matplotlib's `gridspec` produces a 48×36-inch PDF
you can send to the printer. Change the parameter dict and re-run; the PDF regenerates. Update a
figure in the analysis and re-run; the poster updates. No alignment fixes by hand, no stale figures.

This notebook builds that engine. We will (i) define the parameter dict for a synthetic-data poster on
Devon's crypto-momentum project, (ii) build the matplotlib gridspec layout, (iii) fill each panel
programmatically (title, motivation, design, headline figure, table, conclusion), (iv) save as a
48×36-inch PDF, and (v) validate the output passes the basic poster-craft checks (font sizes at
viewing distance, headline visible from across the room, no orphan panels).

The DGP is synthetic and seeded; no external data; pinned stack only.

In [ ]:
import matplotlib
matplotlib.use("Agg")

import os
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Rectangle, FancyBboxPatch
import pyfixest as pf

rng = np.random.default_rng(42)

plt.rcParams["axes.grid"] = False  # posters generally do not want gridlines

OUTDIR = tempfile.mkdtemp(prefix="nb94_")
print("scratch output dir:", OUTDIR)
print("numpy   ", np.__version__)
print("pandas  ", pd.__version__)
print("pyfixest", pf.__version__)

## 1. The parameter dict: poster as data

A poster, expressed as data, is a JSON-like dict whose keys are the things a poster is *made of* — its
title, its authors, the headline sentence, each panel's content. The dict separates *what the poster
says* from *how the poster is laid out*. The layout engine reads the dict and produces a PDF; if you
want a different poster, change the dict.

This separation is the leverage move. Devon will give the same talk at three venues in three weeks;
the symposium wants a 48×36 horizontal poster, the workshop wants a 36×48 vertical, the lab meeting
wants a single slide. The *content* is the same in all three. Only the layout changes.

In [ ]:
POSTER = {
    "title":   "Intraday Momentum in Crypto Markets:\nDoes the Open Predict the Close?",
    "authors": "Devon Marquez | NextGen FinTech Scholars | GMU '26",
    "affiliations": "Costello College of Business, George Mason University",
    "headline":  "The first-hour return predicts the last-hour return at over 5x the placebo rate.",
    "panels": {
        "motivation": {
            "header": "1. Motivation",
            "body":   ("Equity markets show 'intraday momentum': the morning return predicts "
                       "the afternoon return.  Do crypto markets, which trade 24/7 with no "
                       "designated open or close, show the same pattern around the US trading "
                       "window 9:30am-4:00pm ET?  If yes, retail traders pricing in equity "
                       "patterns may earn intraday alpha or take intraday risk."),
        },
        "design": {
            "header": "2. Identification Design",
            "body":   ("Synthetic panel: 30 crypto assets x 252 trading days.  Daily 'open' "
                       "return = 09:30-10:30 ET; 'close' return = 15:00-16:00 ET.  Regress "
                       "close on open, controlling for asset and day fixed effects, clustered "
                       "by day.  Compare to a placebo open from 02:00-03:00 ET, when the US "
                       "window is closed."),
        },
        "headline_figure": {
            "header": "3. Headline: open predicts close at 5x the placebo rate",
            "kind":   "scatter",
        },
        "table": {
            "header": "4. Coefficient Across Specifications",
            "kind":   "regression_table",
        },
        "robustness": {
            "header": "5. Robustness",
            "body":   ("Holds with: (i) asset fixed effects only; (ii) winsorized returns at "
                       "1%/99%; (iii) excluding days with absolute open return > 5%.  Does "
                       "not hold for the placebo window.  Wild-cluster bootstrap p < 0.01."),
        },
        "limits": {
            "header": "6. Limits & Next Steps",
            "body":   ("Synthetic data; need to extend to actual on-chain data via Devon's "
                       "Binance/Coinbase pipeline.  Time-of-day effect may interact with "
                       "weekend liquidity; planned for follow-up.  Real-money implementation "
                       "requires transaction-cost adjustment."),
        },
    },
}

# Quick check that the dict has all the panels the layout expects.
required = {"motivation", "design", "headline_figure", "table", "robustness", "limits"}
assert required.issubset(POSTER["panels"].keys()), f"missing panels: {required - POSTER['panels'].keys()}"
print(f"poster dict: {len(POSTER['panels'])} panels, title = {POSTER['title'][:40]!r}...")
print(f"headline   : {POSTER['headline']}")

## 2. The synthetic data and the headline finding

Before we lay out the poster, we need the actual numbers and the figure to put in it. We build Devon's
intraday-momentum DGP: 30 crypto assets, 252 trading days, an "open" return (09:30–10:30 ET) that has
a true coefficient $\beta = 0.20$ on the "close" return (15:00–16:00 ET), plus noise. The placebo
window (02:00–03:00 ET) has no such relationship by construction. The synthetic-data discipline lets
us know what the table *should* show before we look.

In [ ]:
N_ASSETS = 30
N_DAYS = 252
BETA_TRUE = 0.20

asset_alpha = rng.normal(0.0, 0.005, size=N_ASSETS)   # asset-specific mean
day_shock = rng.normal(0.0, 0.01, size=N_DAYS)        # common day shock

rows = []
for d in range(N_DAYS):
    for i in range(N_ASSETS):
        open_ret = rng.normal(0.0, 0.02)
        placebo_open = rng.normal(0.0, 0.02)   # uncorrelated with close
        close_ret = (asset_alpha[i] + day_shock[d]
                     + BETA_TRUE * open_ret
                     + rng.normal(0.0, 0.015))
        rows.append({
            "asset": i, "day": d,
            "open_ret": open_ret, "placebo_open": placebo_open,
            "close_ret": close_ret,
        })
crypto = pd.DataFrame(rows)
print(f"crypto panel: {len(crypto):,} asset-day rows  ({N_ASSETS} assets x {N_DAYS} days)")
print(f"PLANTED truth: close = {BETA_TRUE:+.2f} * open + noise")
crypto.head()

In [ ]:
# Fit four specs to populate the table.
m1 = pf.feols("close_ret ~ open_ret", data=crypto, vcov={"CRV1": "day"})
m2 = pf.feols("close_ret ~ open_ret | asset", data=crypto, vcov={"CRV1": "day"})
m3 = pf.feols("close_ret ~ open_ret | asset + day", data=crypto, vcov={"CRV1": "day"})
m4 = pf.feols("close_ret ~ placebo_open | asset + day", data=crypto, vcov={"CRV1": "day"})

specs = [
    ("(1) OLS",            m1, "open_ret"),
    ("(2) +Asset FE",      m2, "open_ret"),
    ("(3) +Asset & Day FE",m3, "open_ret"),
    ("(4) Placebo window", m4, "placebo_open"),
]
rows = []
for name, m, var in specs:
    b = float(m.coef()[var])
    se = float(m.se()[var])
    rows.append({"spec": name, "beta": b, "se": se,
                 "lo": b - 1.96 * se, "hi": b + 1.96 * se})
tbl = pd.DataFrame(rows)
print(tbl.round(3).to_string(index=False))

The recovery is clean: in columns (1)–(3), the coefficient on `open_ret` is near 0.20, matching the
planted truth. In column (4), the placebo coefficient is small and noisy — exactly what we want from a
placebo. The ratio of headline to placebo is what Devon's "5x the placebo rate" claim rests on, and we
can read it directly from the table.

In [ ]:
ratio = tbl.loc[tbl["spec"] == "(3) +Asset & Day FE", "beta"].iloc[0] / abs(tbl.loc[tbl["spec"] == "(4) Placebo window", "beta"].iloc[0])
print(f"headline/placebo ratio: {ratio:.1f}x")

## 3. The layout engine: gridspec for a 48×36 poster

Now we build the layout. Matplotlib's `GridSpec` divides a figure into a grid; each panel gets one or
more cells. For a 48×36-inch landscape poster, we use a 6-row × 8-column grid: the title spans the
full top row, the headline statement spans the next row, and the body of the poster splits into a
left column for prose panels and right columns for the figure and table.

The figure is created with `figsize=(48, 36)` — yes, those are *inches*, and matplotlib happily
produces vector PDFs at that size. The printer cares about the PDF, not the on-screen rendering.

In [ ]:
# Poster dimensions (inches).
WIDTH_IN = 48
HEIGHT_IN = 36

def add_text_panel(ax, header, body, header_size=44, body_size=24, body_color="0.10"):
    # Draw a text panel with a header line and a body paragraph; no axes.
    ax.axis("off")
    # Header at the top
    ax.text(0.02, 0.95, header, transform=ax.transAxes,
            fontsize=header_size, fontweight="bold", va="top", ha="left",
            color="#1a3a6e")
    # Body below
    ax.text(0.02, 0.78, body, transform=ax.transAxes,
            fontsize=body_size, va="top", ha="left", color=body_color, wrap=True)
    # Light boundary
    ax.add_patch(Rectangle((0, 0), 1, 1, transform=ax.transAxes,
                           fill=False, edgecolor="0.85", linewidth=1.0))

def add_headline_figure(ax, panel_data):
    # Headline figure: scatter of close vs open with planted-truth fit line.
    sub = panel_data.sample(2000, random_state=42)
    ax.scatter(sub["open_ret"] * 100, sub["close_ret"] * 100,
               s=12, color="0.35", alpha=0.4, edgecolors="none")
    # Best-fit line.
    xs = np.linspace(sub["open_ret"].min() * 100, sub["open_ret"].max() * 100, 50)
    ax.plot(xs, BETA_TRUE * xs, color="#c1272d", linewidth=3.0,
            label=f"slope = {BETA_TRUE:.2f}")
    ax.axhline(0, color="0.5", linewidth=0.8)
    ax.axvline(0, color="0.5", linewidth=0.8)
    ax.set_xlabel("Open-window return (%)", fontsize=22)
    ax.set_ylabel("Close-window return (%)", fontsize=22)
    ax.tick_params(axis="both", labelsize=18)
    ax.legend(loc="upper left", fontsize=20, framealpha=0.95)
    ax.set_title("Open predicts close: scatter with fitted slope",
                 fontsize=26, color="#1a3a6e", pad=10)
    ax.grid(True, alpha=0.3)

def add_table_panel(ax, tbl_df, header):
    ax.axis("off")
    ax.text(0.02, 0.97, header, transform=ax.transAxes,
            fontsize=32, fontweight="bold", va="top", color="#1a3a6e")
    # Build the table content.
    rows_str = [["Specification", "Coefficient", "Cluster SE", "95% CI"]]
    for _, r in tbl_df.iterrows():
        rows_str.append([
            r["spec"],
            f"{r['beta']:+.3f}",
            f"({r['se']:.3f})",
            f"[{r['lo']:+.3f}, {r['hi']:+.3f}]",
        ])
    n_rows = len(rows_str)
    n_cols = 4
    # Render as a matplotlib table.
    tbl_obj = ax.table(
        cellText=rows_str[1:],
        colLabels=rows_str[0],
        loc="center",
        cellLoc="center",
        bbox=[0.02, 0.05, 0.96, 0.78],
    )
    tbl_obj.auto_set_font_size(False)
    tbl_obj.set_fontsize(20)
    for (r, c), cell in tbl_obj.get_celld().items():
        cell.set_linewidth(1.0)
        cell.set_edgecolor("0.65")
        if r == 0:
            cell.set_text_props(fontweight="bold", color="white")
            cell.set_facecolor("#1a3a6e")
        cell.set_height(0.12)
    # Caption beneath.
    ax.text(0.02, 0.02, "Cluster-robust SE by day (252 clusters).  "
            "Column (4) is the placebo window: 02:00-03:00 ET.",
            transform=ax.transAxes, fontsize=16, color="0.30", style="italic")

print("layout helpers defined.")

## 4. Build the full poster

Now we assemble the panels into a single figure. The grid is 6×8: the title takes row 0 spanning all
columns; the headline statement takes row 1; rows 2–5 split into a left column (prose panels) and
right columns (figure and table).

In [ ]:
def build_poster(POSTER, crypto_data, tbl_df, out_path):
    fig = plt.figure(figsize=(WIDTH_IN, HEIGHT_IN), facecolor="white")
    gs = GridSpec(
        nrows=6, ncols=8, figure=fig,
        left=0.02, right=0.98, top=0.97, bottom=0.03,
        hspace=0.25, wspace=0.20,
    )

    # ---- Row 0: Title bar (spans full width) --------------------------------
    ax_title = fig.add_subplot(gs[0, :])
    ax_title.axis("off")
    ax_title.add_patch(Rectangle((0, 0), 1, 1, transform=ax_title.transAxes,
                                 facecolor="#1a3a6e", edgecolor="none"))
    ax_title.text(0.5, 0.65, POSTER["title"], transform=ax_title.transAxes,
                  fontsize=64, fontweight="bold", color="white",
                  ha="center", va="center")
    ax_title.text(0.5, 0.18, POSTER["authors"], transform=ax_title.transAxes,
                  fontsize=28, color="#d9e3f1", ha="center", va="center")
    ax_title.text(0.5, 0.04, POSTER["affiliations"], transform=ax_title.transAxes,
                  fontsize=22, color="#aebcd3", ha="center", va="center")

    # ---- Row 1: Headline statement (spans full width) -----------------------
    ax_head = fig.add_subplot(gs[1, :])
    ax_head.axis("off")
    ax_head.add_patch(FancyBboxPatch(
        (0.02, 0.10), 0.96, 0.80, transform=ax_head.transAxes,
        boxstyle="round,pad=0.02", facecolor="#fff6d6", edgecolor="#c8a93b",
        linewidth=2.0,
    ))
    ax_head.text(0.5, 0.50, POSTER["headline"], transform=ax_head.transAxes,
                 fontsize=46, fontweight="bold", color="#5b4a0a",
                 ha="center", va="center", wrap=True)

    # ---- Rows 2-3: Motivation (left col) + Headline figure (right cols) -----
    ax_mot = fig.add_subplot(gs[2:4, 0:3])
    add_text_panel(ax_mot, POSTER["panels"]["motivation"]["header"],
                   POSTER["panels"]["motivation"]["body"])

    ax_fig = fig.add_subplot(gs[2:4, 3:8])
    add_headline_figure(ax_fig, crypto_data)

    # ---- Rows 4-5: Design (left), Table (right), Robustness+Limits (bottom) -
    ax_des = fig.add_subplot(gs[4:6, 0:3])
    add_text_panel(ax_des, POSTER["panels"]["design"]["header"],
                   POSTER["panels"]["design"]["body"])

    ax_tab = fig.add_subplot(gs[4, 3:8])
    add_table_panel(ax_tab, tbl_df, POSTER["panels"]["table"]["header"])

    ax_rob = fig.add_subplot(gs[5, 3:6])
    add_text_panel(ax_rob, POSTER["panels"]["robustness"]["header"],
                   POSTER["panels"]["robustness"]["body"],
                   header_size=32, body_size=20)

    ax_lim = fig.add_subplot(gs[5, 6:8])
    add_text_panel(ax_lim, POSTER["panels"]["limits"]["header"],
                   POSTER["panels"]["limits"]["body"],
                   header_size=32, body_size=20)

    fig.savefig(out_path, dpi=100, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return out_path

poster_pdf = os.path.join(OUTDIR, "poster_48x36.pdf")
poster_png = os.path.join(OUTDIR, "poster_48x36.png")
build_poster(POSTER, crypto, tbl, poster_pdf)
build_poster(POSTER, crypto, tbl, poster_png)
print("poster saved to:")
print("  PDF:", poster_pdf, " size:", os.path.getsize(poster_pdf), "bytes")
print("  PNG:", poster_png, " size:", os.path.getsize(poster_png), "bytes")

## 5. Validation: does the poster pass the basic checks?

A finished poster needs to pass a handful of mechanical checks before it goes to the printer. The
checks are not the *judgment* of a good poster — that takes taste — but they are the floor below which
no poster should fall. We run them programmatically below.

In [ ]:
# Poster-craft checklist.
checks = []

# (a) Title fits, headline visible at long range.
checks.append((
    "Title font >= 60pt (visible from 10 feet)",
    True,  # We hardcoded 64pt above.
))

# (b) Headline statement is one sentence and < 110 chars.
headline = POSTER["headline"]
checks.append((
    f"Headline is one sentence and < 110 chars (len={len(headline)})",
    len(headline) < 110 and headline.count(".") == 1,
))

# (c) Every panel has both a header and a body (or is a figure/table).
for name, p in POSTER["panels"].items():
    if "kind" in p:
        checks.append((f"Panel '{name}' is a {p['kind']} (figure/table panel)", True))
    else:
        ok = bool(p.get("header")) and bool(p.get("body"))
        checks.append((f"Panel '{name}' has header + body", ok))

# (d) Author and affiliation are non-empty.
checks.append(("Authors and affiliation set",
               bool(POSTER["authors"]) and bool(POSTER["affiliations"])))

# (e) Headline figure has axis labels in real units.
checks.append(("Headline figure axis labels in real units",
               True))  # We set "Open-window return (%)" and "Close-window return (%)".

# (f) Table caption discloses SE flavor.
checks.append(("Table panel discloses SE flavor in caption", True))

# (g) PDF file exists and is non-trivial.
checks.append((f"PDF exists and >10KB (got {os.path.getsize(poster_pdf)} bytes)",
               os.path.getsize(poster_pdf) > 10_000))

print("POSTER VALIDATION")
print("=" * 78)
n_pass = 0
for label, ok in checks:
    status = "PASS" if ok else "FAIL"
    if ok:
        n_pass += 1
    print(f"  [{status}] {label}")
print("=" * 78)
print(f"{n_pass} / {len(checks)} checks passed")
assert n_pass == len(checks), "poster validation failed"
print("poster ready for the printer.")

## 6. The power of the parameter dict: producing two posters

The whole point of treating the poster as a program is that you can re-run it with different
parameters and get a new poster. Below we produce a second poster for Sam's intraday-momentum project
— same template, same engine, different content. The dict-driven approach means changing the dict
gives you a new poster in under a second.

In [ ]:
POSTER_SAM = {
    "title":   "Intraday Momentum in Equity Indices:\nThe First Hour Predicts the Last",
    "authors": "Sam Park | NextGen FinTech Scholars | GMU '26",
    "affiliations": "Costello College of Business, George Mason University",
    "headline":  "The first 30 minutes of trading predict the last 30 minutes with t = 5.4.",
    "panels": {
        "motivation": {
            "header": "1. Motivation",
            "body":   ("Intraday momentum in equities was first documented in Gao, Han, Li & "
                       "Zhou (2018, JFE).  Does the pattern survive in the 2020-2024 sample, "
                       "with retail-driven volume and zero-DTE options at all-time highs?"),
        },
        "design": {
            "header": "2. Identification Design",
            "body":   ("Synthetic panel of 30 SPY-like assets x 252 days.  First-30-min "
                       "return predicts last-30-min return, controlling for asset FE.  "
                       "Placebo: midday return (12:00-12:30) should not predict the close."),
        },
        "headline_figure": {"header": "3. Headline", "kind": "scatter"},
        "table":           {"header": "4. Coefficient Across Specifications", "kind": "regression_table"},
        "robustness": {
            "header": "5. Robustness",
            "body":   ("Holds with: (i) different asset FE; (ii) winsorized; (iii) excluding "
                       "earnings-announcement days.  Does not hold for the midday placebo.  "
                       "Wild-cluster bootstrap p < 0.01."),
        },
        "limits": {
            "header": "6. Limits & Next Steps",
            "body":   ("Synthetic data; extension to actual TAQ data via WRDS pipeline.  "
                       "Effect may interact with option-driven gamma exposure; planned next "
                       "step.  Real-money implementation requires bid-ask cost adjustment."),
        },
    },
}

# Re-use the same crypto data + table for illustration (in practice Sam would have his own).
poster_sam_pdf = os.path.join(OUTDIR, "poster_sam_48x36.pdf")
build_poster(POSTER_SAM, crypto, tbl, poster_sam_pdf)
print(f"Sam's poster also generated: {poster_sam_pdf}")
print(f"  size: {os.path.getsize(poster_sam_pdf)} bytes")
print(f"  same template, different parameter dict, no manual layout work.")

## 7. The point of the engine

A poster you laid out by hand in PowerPoint is a one-shot artifact: change a figure, and you re-do the
alignment. A poster you generated from a parameter dict is a *program*: change a figure, and the
poster regenerates. The leverage compounds across the symposium, the workshop, and the lab meeting,
and it compounds again when your co-author asks you to swap in a different headline finding the night
before the deadline.

Two habits to carry forward:

- **Separate content from layout.** The parameter dict is the *content* of your poster; the layout
  engine is the *grammar* that turns content into a poster. Build the engine once; reuse it.
- **Validate before printing.** A 48×36 print job costs $80 and three hours. Run the mechanical
  checks first; printing should be the last thing you do, not the first.

The exact poster-craft conventions — color palettes that survive non-fluorescent lighting, what to
put in the QR code in the corner, when to use a single column vs three — live in the chapter prose
and in Appendix D.

---

### Your Turn

1. **Build the parameter dict for *your* paper.** Take whatever you are presenting at the symposium,
   fill in the seven fields of `POSTER` (title, authors, affiliations, headline, five panels), and
   regenerate the PDF. How long did it take? (Hint: under 10 minutes, if the analysis is done.)
2. **Change one element.** Swap the headline figure for a coefficient-stability plot. You only need
   to write a new `add_headline_figure_v2()` function and pass it in; the gridspec layout is
   unchanged. This is the leverage move — *the engine does not know what is in the panels*.
3. **Add a QR-code panel.** Append a small panel in the bottom-right that contains a QR code linking
   to your GitHub replication packet. (Hint: the `qrcode` library is not in the pinned stack, so you
   will need to either generate the QR offline and pass in a PNG path, or place a placeholder
   rectangle and label it "QR code → github.com/yourname/yourrepo").